# Layer 15.10 — Experiment Tracking & Hyperparameter Sweeps

**Prerequisites:** Model Training (NB15), Basic PyTorch (NB14)  
**Difficulty:** Intermediate  
**Estimated Time:** 60-90 minutes

**Architectural Layer:** MLOps & Experiment Tracking  
**Toolchain:** Weights & Biases (W&B) · MLflow Tracking · Optuna  
**Objective:** Replace unstructured console prints with professional experiment tracking. Master logging deep learning loss curves, launching distributed hyperparameter sweeps, and securely versioning data/model artifacts.

---

## 📑 Table of Contents
* [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Problem with `print(loss)`](#1-the-problem-with-printloss)
* [2 · Weights & Biases (W&B): Real-Time Tracking](#2-weights--biases-wb-real-time-tracking)
* [3 · Automated Hyperparameter Sweeps](#3-automated-hyperparameter-sweeps)
* [4 · Artifact Logging & Lineage](#4-artifact-logging--lineage)
* [5 · MLflow: The Open Source Alternative](#5-mlflow-the-open-source-alternative)
* [✅ Knowledge Check](#knowledge-check)
* [🔨 Practical Practice](#practical-practice)
* [🎯 Summary & Next Steps](#summary--next-steps)

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors name their models `model_final_v2_really_final.pth` and lose track of which learning rate produced which weights. Seniors treat every model training run as a reproducible, immutable experiment tied to a central database via W&B or MLflow.

**Mental Model:** 
- **Experiment Tracker:** A git ledger, but for data scientists. Instead of commits tracking code changes, runs track metric changes associated with hyperparameter changes.
- **Sweeps:** Systematic searches (Grid, Random, Bayesian) across thousands of configurations, distributed across many GPUs, orchestrated by a central tracking server.

**Common Junior Pitfall:** Writing custom Python `for` loops to test hyperparameter combinations. This is serial, fragile, and unresumable. Use W&B Sweeps or Optuna to manage the parameter space and parallelize workers.


In [ ]:
# Install required libraries
!pip install -q wandb mlflow optuna torch torchvision tqdm pandas
print("✅ Libraries installed.")
    


---
## 1 · The Problem with `print(loss)`

During model development, training runs take hours or days. You cannot stare at terminal standard output for 3 days.

**Common Issues Handled by Experiment Trackers:**
1. **Visualization:** Plotting train/val loss curves live as the model trains.
2. **Comparison:** Overlaying the loss curve of `learning_rate=1e-3` on top of `learning_rate=1e-4` to see which converged faster.
3. **Artifact Retention:** Saving the specific `config.json` alongside the `.pth` weights so the exact setup is perpetually reproducible.
4. **Collaboration:** Sending your manager a dashboard link, not a screenshot of your terminal.


---
## 2 · Weights & Biases (W&B): Real-Time Tracking

**Weights & Biases (W&B)** is the industry standard for cloud-based ML experiment tracking. 
*Note: In a true production environment, you would log in securely (e.g., `wandb login --relogin` using a token).*


In [ ]:
import wandb
import math
import random

# We're running in offline mode so the notebook runs without requiring a W&B account.
# In production, remove this to sync directly to the wandb.ai cloud.
import os
os.environ["WANDB_MODE"] = "offline"

# 1. Initialize an experiment run
# `project` is the folder, `name` is the specific run, `config` tracks hyperparameters.
run = wandb.init(
    project="mlops-2026-curriculum",
    name="baseline-cnn-run",
    config={
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 32,
        "architecture": "ResNet18",
        "optimizer": "AdamW"
    }
)

print(f"🚀 Started W&B Run: {run.name}")

# 2. Simulate a deep learning training loop
epochs = wandb.config.epochs
base_loss = 2.5

for epoch in range(epochs):
    # Simulate decreasing loss and increasing accuracy
    decay = math.exp(-epoch / 3.0)
    train_loss = 0.2 + (base_loss * decay) + random.uniform(0, 0.1)
    val_loss   = 0.3 + (base_loss * decay) + random.uniform(0, 0.2)
    accuracy   = 100 - (train_loss * 20)
    
    # 3. wandb.log() maps metrics to the current step
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "val/loss": val_loss,
        "val/accuracy": accuracy,
        "system/gpu_temp": random.uniform(60, 75)  # Can track hardware health too
    })

print("✅ Training complete. Metrics logged successfully.")

# 4. Finish the run so the agent is closed and synced
wandb.finish()
    


---
## 3 · Automated Hyperparameter Sweeps

Finding the best hyperparameters by guessing and writing nested `for-loops` is unreliable.
A **Sweep** abstracts the search algorithm (Random, Grid, Bayesian). 

The controller dictates which parameters to try next based on past iterations (e.g., adopting Bayesian Optimization to zone in on highly performant hyperparameter clusters).


In [ ]:
# Sweeps require three components:
# 1. A configuration dictionary defining the parameter search space
# 2. A sweep ID (requested from the W&B server)
# 3. An agent worker function to execute combinations

sweep_config = {
    "method": "bayes",  # Bayesian optimization
    "metric": {
        "name": "val/accuracy",
        "goal": "maximize"   
    },
    "parameters": {
        "learning_rate": {
            "min": 1e-4,
            "max": 1e-2
        },
        "dropout": {
            "values": [0.1, 0.2, 0.3, 0.5]
        },
        "batch_size": {
            "distribution": "q_log_uniform_values",
            "q": 8,
            "min": 16,
            "max": 128
        }
    }
}

print("⚙️ Sweep Configuration Initialized:")
import json
print(json.dumps(sweep_config, indent=2))

print("\n💡 In production, you would run:")
print("   sweep_id = wandb.sweep(sweep_config, project='mlops-sweeps')")
print("   wandb.agent(sweep_id, function=my_training_function, count=50)")
    


---
## 4 · Artifact Logging & Lineage

You trained a model, but what exact dataset was it trained on? And where did the output model weights go? 

W&B uses **Artifacts** to implement a directed acyclic graph (DAG) of lineage. You define inputs as "consumed artifacts" and models as "produced artifacts."


In [ ]:
# Let's create some dummy files to simulate artifacts
with open("dataset_v1.csv", "w") as f: f.write("id,feature,label\n1,10.5,1")
with open("model_weights.pth", "w") as f: f.write("0101010010101011111000")

run = wandb.init(project="mlops-2026-curriculum", name="artifact-lineage-demo")

# 1. Start tracking the dataset we are USING (Consumed Artifact)
# In reality, this might be pulling an artifact already created in a previous preprocessing step.
raw_data_artifact = wandb.Artifact(
    name="training_dataset", 
    type="dataset", 
    description="The raw training CSV used for this run"
)
raw_data_artifact.add_file("dataset_v1.csv")
run.use_artifact(raw_data_artifact)

# --- Simulate Training Occurring Here --- #

# 2. Track the model we are PRODUCING (Produced Artifact)
model_artifact = wandb.Artifact(
    name="cnn_classifier", 
    type="model", 
    metadata={"architecture": "CNN", "epochs": 10}
)
model_artifact.add_file("model_weights.pth")
run.log_artifact(model_artifact)

run.finish()

print("✅ Logged dataset as a consumed artifact and model weights as a produced artifact.")
print("   The W&B UI will now render a lineage graph connecting the CSV to the PTH file.")
    


---
## 5 · MLflow: The Open Source Alternative

While W&B excels in SaaS visual tracking, **MLflow Tracking** is standard for teams deploying locally or leveraging Databricks. MLflow uses a tracking server (which can be backed by a simple SQLite file) to log parameters, metrics, and models.


In [ ]:
import mlflow

# 1. Set the Tracking URI (where the DB lives)
mlflow.set_tracking_uri("sqlite:///mlruns.db")

# 2. Set the context of the experiment
mlflow.set_experiment("fraud_detection_baseline")

# 3. Start a tracking run
with mlflow.start_run(run_name="RandomForest_v1"):
    
    # Log parameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    
    # Log metrics over multiple epochs/steps
    for step in range(5):
        loss = 1.0 / (step + 1)
        mlflow.log_metric("train_loss", loss, step=step)
        
    # Log a local file as an artifact
    with open("feature_importance.txt", "w") as f:
        f.write("Income: 0.45\nAge: 0.3")
    mlflow.log_artifact("feature_importance.txt", artifact_path="eval")

print("✅ Successfully logged experiment to local MLflow SQLite database (mlruns.db).")
print("   Run `mlflow ui --backend-store-uri sqlite:///mlruns.db` to view the dashboard in your browser.")
    


---
## ✅ Knowledge Check

1. **Why is Bayesian Optimization (`method: "bayes"`) usually preferred over Grid Search for deep learning hyperparameter sweeps?**
<details>
<summary>Answer</summary>
Grid Search tries every possible combination, which takes exponentially long as parameters increase (curse of dimensionality). Deep Learning runs are too expensive computationally to exhaustively test everything. Bayesian Optimization uses past results to map probability distributions, spending compute ONLY in regions that show promise.
</details>

2. **In W&B Artifacts, what is the difference between `run.use_artifact()` and `run.log_artifact()`?**
<details>
<summary>Answer</summary>
`use_artifact()` documents that your training script read (consumed) a specific version of a dataset or intermediate model, pulling it into the environment if needed. `log_artifact()` saves and versions the output strictly generated by the current run, creating an immutable lineage graph.
</details>

3. **What is a major organizational difference between MLflow and W&B?**
<details>
<summary>Answer</summary>
Weights & Biases is primarily a hosted SaaS application known for excellent visuals and collaboration. MLflow is an open-source framework you must host and scale yourself (unless utilizing managed Databricks), focusing deeply on standardizing the whole MLOps lifecycle from Tracking to Registry to Deployment.
</details>

---

## 🔨 Practical Practice

**Exercise 1: Extend W&B Logging** 
In Cell 2, the `wandb.log` function tracks metrics over time. Modify the dictionary to track a new metric named `train/learning_rate` that decays linearly by 10% each epoch.

**Exercise 2: Log a Confusion Matrix**
Weights and Biases supports rich media logging. Review the W&B docs on how to log an image or a table. Write pseudo-code using `wandb.Table` to log a dataset sample of 3 rows with columns: ["Input", "Prediction", "Confidence"].

**Exercise 3: MLflow Tags**
MLflow allows adding searchable tags to runs. Using the `mlflow` library in Cell 5, add a command that sets a tag `{"team": "risk", "environment": "dev"}` on the current run. 

---

## 🎯 Summary & Next Steps

### We accomplished:
1. Replaced static CLI printouts with dynamic, visual metric tracking using **W&B**.
2. Designed scalable search strategies for determining hyperparameters utilizing **Sweeps**.
3. Connected raw data inputs to final model weights using immutable **Artifact Lineage Graphs**.
4. Logged parameters securely into a local SQLite database utilizing **MLflow Tracking**.

### Next Path →
With mastering experimentation, you are ready to move towards textual sequence processing models and advanced tokenization logic.
**Proceed to → `16_01_text_preprocessing_tokenization.ipynb`**.
